<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/transformers_agents_multiagents/Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Implementing the Transformer Architecture

In [ ]:
!pip install bertviz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 89.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 9.0 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer

In [ ]:
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
text = "time flies like an arrow"

In [ ]:
inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

In [ ]:
inputs

{'input_ids': tensor([[ 2051, 10029,  2066,  2019,  8612]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

We create a dense embedding using the torch.nn.Embedding layer which acts as a lookup table for each input ID.

In [ ]:
from torch import nn
from transformers import AutoConfig

In [ ]:
config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
token_emb

Embedding(30522, 768)

Above, we used AutoConfig class to load the config.json file associated with the bert-base-uncased checkpoint, from the output above, each input ID will be mapped to one out of 30522 embedding vectors stored in nn.Embedding, each with a size of 758. The token embeddings at this point are independent of their context. Remember that the layer acts as our lookup table. Lets then generate the embeddings by feeding in the input IDs


In [ ]:
inputs_embeds = token_emb(inputs.input_ids)
inputs_embeds.size()

torch.Size([1, 5, 768])

Output is in the form of: \[batch_size, sequence_length,hidden_state_dimension].


 The next steps implement some form of self attention, we've kept things simple here and instead of projecting the positional encodings into query, key and value vectors, we've kept them equal, for now(key, value and query vectors are generated by applying independent weight matrices to the embeddings(?).
We then move on to step 2, which is to  calculate the attention scores using the dot product as the similarity function

In [ ]:
import torch
from math import sqrt

In [ ]:
def attention_dot_product(query, key):
    return torch.bmm(query, key.transpose(1, 2))

def scaled_attention_dot_product(query, key):
    """
    scale dot_product by size of embedding vectors so we don't get too many
    large numbers during training.
    """
    dim_k = key.size(-1)
    return attention_dot_product(query, key) / sqrt(dim_k)

In [ ]:
query = key = value = inputs_embeds
scores = scaled_attention_dot_product(query, key)
scores.size()

torch.Size([1, 5, 5])

the torch.bmm() function performs batch matrix-matrix product which simplifies the computation of the attention scores **where the query and key vectors have the shape** \[batch_size, sequence_length, hidden_dimension].

 If we ignored the batch dimensions, we could calculate the dot product between each query and key vector by simply transposing the key tensor to have the same shape \[hidden_dimmension, sequence_length] and then using the matrix product to collect all the dot products in a \[sequence_length, sequence_length] matrix(?). Since we want to do thhis for all sequences in the batch independently, we use torch.bmm **which takes two batches of matrices** and **multiplies each matrix from the first batch with the corresponding matrix in the second batch** -
 `torch.bmm(batch_of_matrix1, batch_of_matrices2)

In [ ]:
# Step 3: Compute attention weights
import torch.nn.functional as F

In [ ]:
weights = F.softmax(scores, dim=-1)
weights.sum(dim=-1)

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)

In [ ]:
# Step 4: Multiply attention weights by the values
attn_outputs = torch.bmm(weights, value)
attn_outputs.shape

torch.Size([1, 5, 768])

In [ ]:
attn_outputs

tensor([[[ 1.2343,  0.3515, -0.1723,  ...,  2.3884,  0.5072, -0.9034],
         [ 1.0542, -2.5241, -0.0386,  ...,  0.0211, -0.2457, -0.0740],
         [-0.6814, -0.1224, -0.2143,  ...,  1.2551,  1.1538,  0.3626],
         [ 0.1120,  0.1298, -1.2425,  ..., -0.4003, -0.0359,  2.1364],
         [-0.7675,  0.0662,  1.7835,  ..., -1.6175, -2.2865,  0.0409]]],
       grad_fn=<BmmBackward0>)

In [ ]:
# Wrapping the steps into a function
def scaled_dot_product_attention(query, key, value):
    def attention_dot_product(query, key):
        return torch.bmm(query, key.transpose(1, 2))

    def scaled_attention_dot_product(query, key):
        """
        scale dot_product by size of embedding vectors so we don't get too many
        large numbers during training.
        """
        dim_k = key.size(-1)
        return attention_dot_product(query, key) / sqrt(dim_k)

    scores = scaled_attention_dot_product(query, key)
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value)

Our attention mechanism with equal query and key vectors will assign a very large score to **identical words** in the context, and in particular to the corrent word itself: the dot product of a query with itself is always 1. But in practice, the meaning of a word will be better informed by **complementary words** in the context than by identical words, eg the meaning of "flies" is better defined by incorporating information from "time" and "arrow" than by another mention of "flies"

For the above reason, let's allow the model to create a different set of vectors for the query, key and value of a token by using 3 different linear projections to project our initial token vector into 3 different spaces.

#### Multi-headed attention
In the simple example above, we only used the embeddings "as is" to compute the attention scores and weights, but in practice, the self attention layer applies 3 independent linear transformations to each embedding to generate the query, key and value vectors.These transformations project the embeddings and each projection carries its own set of learnable parameters, which allows the self attention layer to focus on different semantic apects of the sequence.

Its also beneficial to have multiple sets of linear projections, each one representing a so-called "attention head". **The reason why we need more than one attention head is that the softmax of one head tend to focus on mostly one aspect of similarity.** Having several heads allows the model to focus on several aspects at once. For instance, one head can focus on subject-verb interaction, whereas another finds nearby adjectives. We don't handcraft these relations into the model and they are fully learned from the data. This can be likened to filters in CNNs where one filter can be responsible for detecting faces and another one finds wheels of cars in images.

In [ ]:
# A single Attention head
class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim):
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden_state):
        attn_outputs = scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state)
        )
        return attn_outputs

We've initialized 3 independent linear layers that can apply matrix multiplication to the embedding vectors to produce tensors of shape \[batch_size, sequence_length, head_dimensions] where head_dim is the number of dimensions we're projecting into. Although head_dim does not have to be smaller than the number of embedding dimensions of the tokens(embed_dim), in practice, it is choosen to be a multiple of embed_dim so that the computation across each head is constant. Eg, BERT has 12 attention heads, so the dimension of each head is 768/12 = 64.

In [ ]:
# Now that we have a single attention head, we concatenate the outputs of each
# one to implement the full multi-head attention layer

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        embed_dim = config.hidden_size
        num_heads = config.num_attention_heads
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )
        self.output_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, hidden_state):
        x = torch.cat([h(hidden_state) for h in self.heads], dim=-1)
        x = self.output_linear(x)
        return x


The concatenated output from the attention heads is also fed through a final linear layer to produce an output tensor of shape \[batch_size, sequence_size, hidden_dimension] that is suitable for the feed-forward network downstream.

In [ ]:
multihead_attn = MultiHeadAttention(config)

In [ ]:
attn_output = multihead_attn(inputs_embeds)

In [ ]:
attn_output.size()

torch.Size([1, 5, 768])

In [ ]:
from bertviz import head_view
from transformers import AutoModel

In [ ]:

model = AutoModel.from_pretrained(model_ckpt, output_attentions=True)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:


sentence_a = "time flies like an arrow"
sentence_b = "fruit files like a banana"

In [ ]:
viz_inputs = tokenizer(sentence_a, sentence_b, return_tensors="pt")
attention = model(**viz_inputs).attentions
sentence_b_start = (viz_inputs.token_type_ids == 0).sum(dim=1)
tokens = tokenizer.convert_ids_to_tokens(viz_inputs.input_ids[0])

In [ ]:
head_view(attention, tokens, sentence_b_start, heads=[8])

<IPython.core.display.Javascript object>



One thing we can see from the above visualization is that the attention weights are strongest between words that belong to the same sentence, this suggests that BERT can tell that it should attend to words in the same sentence.

#### the Feed Forward Layer
This layer is just a simple two layer fully connected neural network, but with a twist: instead of processing the whole sequence of embeddings as a single vector, it processes each embedding independently. For this reason, it is often referred to as a position-wise feed-forward layer.(?)It may also be referred to as a one-dimensional convolution with a kernel size of 1.
A rule of thumb is for the hidden size of the first layer to be four times the size of the embeddings, and a GELU activation function is commonly used. this is where most of the capacity and memorization is hypothesized to happen and it's the part that is most often scaled when scaling up models

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x

The feed-forward layer such as nn.Linear is usually applied to a tensor of shape(batch_size, input_dim), where it acts on each element of the batch dimension independently. This is true for any dimension except the last one, so when we pass a tensor of shape(batch_size, sequence_length, hidden_dimension) the layer is applied to all token embeddings of the batch and sequence independently, which is exactly what we want.(?)

In [ ]:
feed_forward = FeedForward(config)
ff_outputs = feed_forward(attn_outputs)
ff_outputs.size()

torch.Size([1, 5, 768])

We now have all ingredients to create a fully fledged transformer encoder layer!

Let's then stick together our building blocks as follows:

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
        self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)

    def forward(self, x):
        # Apply layer normalization and then copy input into query, key and value
        hidden_state = self.layer_norm_1(x)
        # Apply attention with a skip connection
        x = x + self.attention(hidden_state)
        # Apply feed-forward layer with a skip connection
        x = x + self.feed_forward(self.layer_norm_2(x))
        return x

In [ ]:
encoder_layer = TransformerEncoderLayer(config)

In [ ]:
inputs_embeds.shape, encoder_layer(inputs_embeds).size()

(torch.Size([1, 5, 768]), torch.Size([1, 5, 768]))

We've implemented our very first transformer encoder layer from scratch, however, there is a caveat with the way we set up the encoder layers: They are totally invariant to the position of the tokens. Since the multi-head attention layer is effectively a fancy weighted sum, the information on token position is lost.

#### Positional Embedding

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embeddings = nn.Embedding(config.vocab_size,
                                             config.hidden_size)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings,
                                                config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout()

    def forward(self, input_ids):
        # create position ids for input sequence
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0)
        # create token and position embeddings
        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)
        # combine token and position embeddings
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

In [ ]:
embedding_layer = Embeddings(config)
embedding_layer(inputs.input_ids).size()

torch.Size([1, 5, 768])

Let's put all of the above together by building a full transformer encoder combining the embeddings with the encoder layers


In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embeddings = Embeddings(config)
        self.layers = nn.ModuleList([TransformerEncoderLayer(config)
                        for _ in range(config.num_hidden_layers)])

    def forward(self, x):
        x = self.embeddings(x)
        for layer in self.layers:
            x = layer(x)
        return x

In [ ]:
encoder = TransformerEncoder(config)
encoder(inputs.input_ids).size()

torch.Size([1, 5, 768])

What we have built so far is the body on an encoder network. Given that transformer models are usually divided into a task-independent body and a task-specific head, if we wish to build, say a text classifier, we will need to attach a classification head to that body. We have a hidden state for each token, but we only need to make one prediction. There are several options to approach this. Traditionally, the first token in such models is used for the prediction and we can attach a dropout and a linear layer to make the classification prediction.
The ff class extends the existing encoder for sequence classification


In [ ]:
class TransformerForSequenceClassification(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.encoder = TransformerEncoder(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, x):
        x = self.encoder(x)[:, 0, :] #select hidden state of [CLS] token
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [ ]:
# define how may classes we want to predict
config.num_labels = 3
encoder_classifier = TransformerForSequenceClassification(config)
encoder_classifier(inputs.input_ids).size()

torch.Size([1, 3])

In [ ]:
encoder_classifier(inputs.input_ids)

tensor([[-0.3994, -1.3918,  0.6890]], grad_fn=<AddmmBackward0>)

### The Decoder
Let's take a look at the modifications we need to make to include masking in our self attention layer, and leave the implementation of the encoder-decoder attention layer as a homework problem.

# To Do: Implement The encoder-decoder attention layer

The trick with masked self-attention is to introduce a _mask matrix_ with ones on the lower diagonal and zeros above

In [ ]:
seq_len = inputs.input_ids.size(-1)
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
# torch.tril creates a lower trianqulare matrix
mask

Once we have the mask matrix, we can prevent each attention head from peeking at future tokens by using Tensor.masked_fill() to replace all the zeros with negative infinity.

In [ ]:
scores.masked_fill(mask==0, -float("inf"))

tensor([[[27.9599,    -inf,    -inf,    -inf,    -inf],
         [ 0.4384, 25.8963,    -inf,    -inf,    -inf],
         [ 1.8552,  0.4026, 28.0172,    -inf,    -inf],
         [-1.0078,  1.0503, -0.2549, 30.0066,    -inf],
         [-0.4316, -0.6743, -2.7530, -1.7299, 27.6859]]],
       grad_fn=<MaskedFillBackward0>)

In [ ]:
scores

tensor([[[27.9599,  0.4384,  1.8552, -1.0078, -0.4316],
         [ 0.4384, 25.8963,  0.4026,  1.0503, -0.6743],
         [ 1.8552,  0.4026, 28.0172, -0.2549, -2.7530],
         [-1.0078,  1.0503, -0.2549, 30.0066, -1.7299],
         [-0.4316, -0.6743, -2.7530, -1.7299, 27.6859]]],
       grad_fn=<DivBackward0>)

By setting the upper values to negative infinity, we can guarantee that the attention weights are all zero once we take the softmax over the scores because $e^{-\infty} = 0$(softmax calculates the normalized exponential).
Let's include this masking behaviour with a small change to our scaled dot-product attention funtion implemented earlier

In [ ]:
def scaled_dot_product_attention(query, key, value):
    def attention_dot_product(query, key):
        return torch.bmm(query, key.transpose(1, 2))

    def scaled_attention_dot_product(query, key):
        """
        scale dot_product by size of embedding vectors so we don't get too many
        large numbers during training.
        """
        dim_k = key.size(-1)
        return attention_dot_product(query, key) / sqrt(dim_k)

    scores = scaled_attention_dot_product(query, key)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value)

### ToDo: Build the decoder network. Use [Andrej Karpathy's implementation of minGPT](https://github.com/karpathy/minGPT) as guide.
